In [1]:
!pip install nltk

import nltk
from nltk.corpus import brown
from collections import defaultdict, Counter
import math
import random

nltk.download("brown")
nltk.download("universal_tagset")


[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Unzipping corpora/brown.zip.
[nltk_data] Downloading package universal_tagset to /root/nltk_data...
[nltk_data]   Unzipping taggers/universal_tagset.zip.


True

In [2]:
sentences = brown.tagged_sents(tagset="universal")

print("Total sentences:", len(sentences))
print("Example:", sentences[0])


Total sentences: 57340
Example: [('The', 'DET'), ('Fulton', 'NOUN'), ('County', 'NOUN'), ('Grand', 'ADJ'), ('Jury', 'NOUN'), ('said', 'VERB'), ('Friday', 'NOUN'), ('an', 'DET'), ('investigation', 'NOUN'), ('of', 'ADP'), ("Atlanta's", 'NOUN'), ('recent', 'ADJ'), ('primary', 'NOUN'), ('election', 'NOUN'), ('produced', 'VERB'), ('``', '.'), ('no', 'DET'), ('evidence', 'NOUN'), ("''", '.'), ('that', 'ADP'), ('any', 'DET'), ('irregularities', 'NOUN'), ('took', 'VERB'), ('place', 'NOUN'), ('.', '.')]


In [4]:
sentences = list(sentences)

random.seed(42)
random.shuffle(sentences)

n = len(sentences)

train_sents = sentences[:int(0.8*n)]
dev_sents   = sentences[int(0.8*n):int(0.9*n)]
test_sents  = sentences[int(0.9*n):]

print("Train:", len(train_sents), "Dev:", len(dev_sents), "Test:", len(test_sents))


Train: 45872 Dev: 5734 Test: 5734


In [5]:
def preprocess(sents):
    data = []
    for sent in sents:
        new_sent = []
        for word, tag in sent:
            word = word.lower()
            new_sent.append((word, tag))
        data.append(new_sent)
    return data

train_sents = preprocess(train_sents)
dev_sents   = preprocess(dev_sents)
test_sents  = preprocess(test_sents)

vocab = set(word for sent in train_sents for word, tag in sent)
tags = set(tag for sent in train_sents for _, tag in sent)

print("Vocabulary size:", len(vocab))
print("Tags:", tags)


Vocabulary size: 45153
Tags: {'ADV', '.', 'NUM', 'CONJ', 'DET', 'ADP', 'NOUN', 'PRT', 'X', 'PRON', 'ADJ', 'VERB'}


In [6]:
transition_counts = defaultdict(Counter)

emission_counts = defaultdict(Counter)

tag_counts = Counter()

for sent in train_sents:
    prev = "<START>"
    tag_counts[prev] += 1
    for word, tag in sent:
        transition_counts[prev][tag] += 1
        emission_counts[tag][word] += 1
        tag_counts[tag] += 1
        prev = tag
    transition_counts[prev]["<END>"] += 1


In [7]:
all_tags = list(tags) + ["<START>", "<END>"]

def transition_prob(prev_tag, tag):
    numerator = transition_counts[prev_tag][tag] + 1
    denominator = sum(transition_counts[prev_tag].values()) + len(all_tags)
    return math.log(numerator / denominator)

def emission_prob(tag, word):
    numerator = emission_counts[tag][word] + 1
    denominator = sum(emission_counts[tag].values()) + len(vocab)
    return math.log(numerator / denominator)


In [8]:
def viterbi(words):
    words = [w.lower() for w in words]

    V = [{}]
    backpointer = [{}]

    for tag in tags:
        V[0][tag] = transition_prob("<START>", tag) + emission_prob(tag, words[0])
        backpointer[0][tag] = "<START>"

    for t in range(1, len(words)):
        V.append({})
        backpointer.append({})
        for tag in tags:
            best_prev = max(tags, key=lambda prev: V[t-1][prev] + transition_prob(prev, tag))
            V[t][tag] = V[t-1][best_prev] + transition_prob(best_prev, tag) + emission_prob(tag, words[t])
            backpointer[t][tag] = best_prev

    best_last = max(tags, key=lambda tag: V[-1][tag] + transition_prob(tag, "<END>"))

    output = [best_last]
    for t in range(len(words)-1, 0, -1):
        output.insert(0, backpointer[t][output[0]])

    return output


In [9]:
def evaluate(dataset):
    correct = 0
    total = 0
    for sent in dataset:
        words = [w for w, t in sent]
        gold =  [t for w, t in sent]

        pred = viterbi(words)

        for p, g in zip(pred, gold):
            if p == g:
                correct += 1
            total += 1
    return correct / total

accuracy = evaluate(test_sents)
print("Test accuracy:", accuracy)


Test accuracy: 0.9388657221545926


In [10]:
test_sentence = ["I", "love", "machine", "learning"]
print(list(zip(test_sentence, viterbi(test_sentence))))


[('I', 'PRON'), ('love', 'VERB'), ('machine', 'NOUN'), ('learning', '.')]
